# MMR로 관련성과 다양성을 함께 고려해 선택하기

이 노트북은 공개 PDF 6건과 Ch11의 30문항을 사용합니다. 첫 셀부터 순서대로 실행합니다.


## 입력 준비

최대 주변 관련성(Maximal Marginal Relevance, MMR)을 수식 그대로 직접 구현합니다.

### 관련성과 중복 억제를 한 단계씩 계산하기

MMR은 질문 관련성에서 이미 고른 문서와의 최대 유사도를 뺀 점수로 다음 후보를 선택합니다. `lambda_mult`가 1에 가까우면 관련성, 0에 가까우면 기존 선택과의 차이를 더 크게 봅니다. 이번 λ=0.2 출력에는 질문 관련성 0인 청크도 포함되므로 고유 부모 수와 관련성 점수를 함께 확인합니다.


In [1]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "labs/part3/ch12-retrieval-tuning").exists())
data = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-documents.json").read_text())
queries = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-queries.json").read_text())
chunks, parents = data["chunks"], {item["doc_id"]: item for item in data["parents"]}
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
matrix = vectorizer.fit_transform([item["text"] for item in chunks])
print(f"부모 문서 {len(parents)}건 · 자식 청크 {len(chunks)}개 · 평가 질문 {len(queries)}개")

부모 문서 6건 · 자식 청크 175개 · 평가 질문 30개


## MMR 함수

λ가 1에 가까우면 질문 관련성을, 0에 가까우면 이미 고른 청크와의 차이를 더 크게 봅니다.


In [2]:
def mmr(query_vector, candidate_matrix, k=5, lambda_mult=0.5):
    relevance = cosine_similarity(query_vector, candidate_matrix)[0]
    selected, remaining = [], list(range(candidate_matrix.shape[0]))
    while remaining and len(selected) < k:
        if not selected:
            choice = max(remaining, key=lambda i: relevance[i])
        else:
            redundancy = cosine_similarity(candidate_matrix[remaining], candidate_matrix[selected]).max(axis=1)
            values = lambda_mult * relevance[remaining] - (1 - lambda_mult) * redundancy
            choice = remaining[int(np.argmax(values))]
        selected.append(choice)
        remaining.remove(choice)
    return selected

## λ 값 비교

같은 질문과 같은 후보 행렬에서 λ만 바꿉니다.


In [3]:
query = next(item for item in queries if item["qid"] == "Q001")
assert query["split"] == "dev"
question = query["question"]
qv = vectorizer.transform([question])
for value in (1.0, 0.5, 0.2):
    chosen = mmr(qv, matrix, k=6, lambda_mult=value)
    parent_ids = [chunks[idx]["parent_id"] for idx in chosen]
    print(f"lambda={value:.1f}", parent_ids, "고유 부모", len(set(parent_ids)))

lambda=1.0 ['E01', 'E01', 'E02', 'E01', 'E01', 'E01'] 고유 부모 2
lambda=0.5 ['E01', 'E02', 'E01', 'E01', 'H01', 'M02'] 고유 부모 4
lambda=0.2 ['E01', 'E01', 'E02', 'E01', 'H01', 'H02'] 고유 부모 4


## 관련성 점수도 함께 읽기

다양성이 커져도 질문과 무관한 결과가 선택될 수 있으므로 점수를 확인합니다.


In [4]:
relevance = cosine_similarity(qv, matrix)[0]
chosen = mmr(qv, matrix, k=6, lambda_mult=0.2)
for idx in chosen:
    print(chunks[idx]["chunk_id"], f"관련성={relevance[idx]:.3f}")

E01-P01-C01 관련성=0.421
E01-P02-C04 관련성=0.172
E02-P05-C01 관련성=0.194
E01-P05-C04 관련성=0.000
H01-P04-C06 관련성=0.000
H02-P01-C02 관련성=0.000
